# Apply Curation

Applica le sostituzioni individuate da `answer_curation.ipynb`.

**Input** (tutti file esistenti, nessuno viene sovrascritto):
- `top100_merged.parquet` — top-100 originali (1.000 query)
- `top100_candidates.parquet` — top-100 delle 5.000 candidate
- `passage_entities.parquet` + `curation_chunks/*.parquet` — entità dei passaggi
- `query_embeddings.npy` — embedding delle 1.000 query originali
- `queries_subset.jsonl` — metadati delle 1.000 query originali
- `curation_results.jsonl` — mapping delle 344 sostitute valide
- `qa_all_entities.jsonl` — pool completo delle query NQ (per metadati sostitute)

**Output** (file nuovi):
- `top100_curated.parquet`
- `passage_entities_curated.parquet`
- `query_embeddings_curated.npy`
- `queries_curated.jsonl`

In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl

OUTPUT_DIR  = Path("data/NQ_answer")
QUESTION_DIR = Path("data/NQ_question")

## 1. Load Data

In [ ]:
# --- Original dataset ---
top100 = pl.read_parquet(OUTPUT_DIR / "top100_merged.parquet")
print(f"top100_merged:        {top100.height:,} rows, {top100['query_id'].n_unique()} queries")

# --- Candidate dataset ---
cand_top100 = pl.read_parquet(OUTPUT_DIR / "top100_candidates.parquet")
print(f"top100_candidates:    {cand_top100.height:,} rows, {cand_top100['query_id'].n_unique()} queries")

# --- Passage entities: original + curation chunks ---
psg_ent_orig = pl.read_parquet(OUTPUT_DIR / "passage_entities.parquet")
curation_chunks = pl.read_parquet(OUTPUT_DIR / "curation_chunks/")
psg_ent_all = pl.concat([psg_ent_orig, curation_chunks]).unique(subset=["id"])
print(f"passage_entities:     {psg_ent_orig.height:,} (original) + {curation_chunks.height:,} (curation) = {psg_ent_all.height:,} (unique)")

# --- Embeddings ---
orig_emb = np.load(str(OUTPUT_DIR / "query_embeddings.npy"))
print(f"query_embeddings:     {orig_emb.shape}")

# --- Curation mapping ---
with open(OUTPUT_DIR / "curation_results.jsonl", encoding="utf-8") as f:
    subs = [json.loads(line) for line in f]
print(f"curation_results:     {len(subs)} substitutes")

## 2. Identify Queries to Replace

Ricalcoliamo le query problematiche direttamente dai dati: sono quelle che hanno
almeno un passaggio con 0 entità nella top-100 originale.

In [ ]:
# Join top100 with entity counts
ent_counts = psg_ent_orig.select("id", pl.col("qids").list.len().alias("n_qids"))

top100_ent = top100.join(ent_counts, left_on="passage_id", right_on="id", how="left")

# Queries with at least one 0-entity passage
queries_to_replace = set(
    top100_ent.filter(pl.col("n_qids") == 0)["query_id"].unique().to_list()
)

n_total = top100["query_id"].n_unique()
print(f"Queries to replace: {len(queries_to_replace)} / {n_total}")
assert len(queries_to_replace) == len(subs), (
    f"Mismatch: {len(queries_to_replace)} bad queries vs {len(subs)} substitutes"
)

## 3. Apply Substitution

Ogni sostituta **eredita il query_id** della query che rimpiazza, così il range 0–999 resta intatto.

In [ ]:
bad_qids = sorted(queries_to_replace)

# Mapping: candidate local_query_id → inherited query_id
sub_lid_to_qid = {
    s["local_query_id"]: bqid
    for bqid, s in zip(bad_qids, subs)
}

# 3a. Keep good original rows (unchanged)
top100_good = top100.filter(~pl.col("query_id").is_in(bad_qids))

# 3b. Substitute rows: filter + remap query_id
sub_local_ids = list(sub_lid_to_qid.keys())
top100_subs = (
    cand_top100
    .filter(pl.col("query_id").is_in(sub_local_ids))
    .with_columns(
        pl.col("query_id").replace(sub_lid_to_qid).alias("query_id")
    )
)

# 3c. Concatenate
top100_curated = pl.concat([top100_good, top100_subs]).sort("query_id", "rank")

print(f"top100_curated: {top100_curated.height:,} rows, "
      f"{top100_curated['query_id'].n_unique()} queries")
print(f"  good original: {top100_good.height:,} rows ({top100_good['query_id'].n_unique()} queries)")
print(f"  substituted:   {top100_subs.height:,} rows ({top100_subs['query_id'].n_unique()} queries)")

## 4. Curated Passage Entities

Filtriamo solo i passaggi effettivamente referenziati dal top-100 curato.

In [ ]:
curated_pids = top100_curated["passage_id"].unique()
psg_ent_curated = psg_ent_all.filter(pl.col("id").is_in(curated_pids))

# Check: every passage in curated top-100 must have entity data
missing = curated_pids.filter(~curated_pids.is_in(psg_ent_curated["id"]))
print(f"Passages in curated top-100:   {curated_pids.len():,}")
print(f"Passages with entity data:     {psg_ent_curated.height:,}")
print(f"Missing entity data:           {missing.len()}")

if missing.len() > 0:
    print("\n⚠ WARNING: some passages lack entity data!")

## 5. Curated Query Embeddings

Per le 656 query buone usiamo gli embedding originali.  
Per le 344 sostitute serve ricodificare con Contriever.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModel

# Load substitute questions
with open(QUESTION_DIR / "qa_all_entities.jsonl", encoding="utf-8") as f:
    all_queries = [json.loads(line) for line in f]

sub_questions = [all_queries[s["original_query_id"]]["question"] for s in subs]
print(f"Substitute questions to encode: {len(sub_questions)}")

# Mean-pooling (same as answer_preparation & embedding.ipynb)
def mean_pool(outputs, attention_mask):
    hidden = outputs.last_hidden_state  # (B, L, D)
    mask = attention_mask.unsqueeze(-1).float()  # (B, L, 1)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1)

# Encode with Contriever
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("facebook/contriever")
model = AutoModel.from_pretrained("facebook/contriever").to(device).eval()

BATCH_SIZE = 128
sub_embs = []
t0 = time.time()

with torch.no_grad():
    for i in range(0, len(sub_questions), BATCH_SIZE):
        batch = sub_questions[i : i + BATCH_SIZE]
        tok = tokenizer(batch, padding=True, truncation=True,
                        max_length=512, return_tensors="pt").to(device)
        out = model(**tok)
        emb = mean_pool(out, tok["attention_mask"])
        sub_embs.append(emb.cpu().numpy())

sub_embs = np.concatenate(sub_embs, axis=0)  # (344, 768)
print(f"Encoded {sub_embs.shape[0]} queries → {sub_embs.shape} in {time.time()-t0:.1f}s")

# Free GPU
del model
torch.cuda.empty_cache()

# Build curated embeddings: copy original, overwrite bad slots
curated_emb = orig_emb.copy()
for s, new_qid in zip(subs, bad_qids):
    idx_in_sub = [x["local_query_id"] for x in subs].index(s["local_query_id"])
    curated_emb[new_qid] = sub_embs[idx_in_sub]

print(f"curated_emb shape: {curated_emb.shape}")

## 6. Curated Query Metadata

Ricostruiamo il jsonl con i metadati completi.  
Le query sostituite vengono marcate con `"curated": true`.

In [ ]:
# Load original query metadata
with open(OUTPUT_DIR / "queries_subset.jsonl", encoding="utf-8") as f:
    orig_queries_list = [json.loads(line) for line in f]

curated_queries = list(orig_queries_list)  # shallow copy of the list

for s, new_qid in zip(subs, bad_qids):
    src = all_queries[s["original_query_id"]]
    curated_queries[new_qid] = {
        "question":           src["question"],
        "answers":            src["answers"],
        "question_qids":      src["question_qids"],
        "answer_variant_qids": src["answer_variant_qids"],
        "original_query_id":  s["original_query_id"],
        "curated":            True,
    }

n_curated = sum(1 for q in curated_queries if q.get("curated"))
print(f"queries_curated: {len(curated_queries)} total, {n_curated} substituted")

## 7. Save

In [ ]:
top100_curated.write_parquet(OUTPUT_DIR / "top100_curated.parquet")
psg_ent_curated.write_parquet(OUTPUT_DIR / "passage_entities_curated.parquet")
np.save(str(OUTPUT_DIR / "query_embeddings_curated.npy"), curated_emb)

with open(OUTPUT_DIR / "queries_curated.jsonl", "w", encoding="utf-8") as f:
    for q in curated_queries:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print("Saved:")
print(f"  {OUTPUT_DIR / 'top100_curated.parquet'}")
print(f"  {OUTPUT_DIR / 'passage_entities_curated.parquet'}")
print(f"  {OUTPUT_DIR / 'query_embeddings_curated.npy'}")
print(f"  {OUTPUT_DIR / 'queries_curated.jsonl'}")

## 8. Sanity Check

Verifica che nel dataset curato non ci siano passaggi con 0 entità.

In [ ]:
# Join curated top100 with entity counts
ent_check = psg_ent_curated.select("id", pl.col("qids").list.len().alias("n_qids"))
curated_with_ent = top100_curated.join(ent_check, left_on="passage_id", right_on="id", how="left")

zero_remaining = curated_with_ent.filter(
    (pl.col("n_qids") == 0) | pl.col("n_qids").is_null()
).height

print(f"{'='*50}")
print(f"SANITY CHECK")
print(f"{'='*50}")
print(f"Total rows:              {curated_with_ent.height:,}")
print(f"Total queries:           {curated_with_ent['query_id'].n_unique()}")
print(f"0-entity rows remaining: {zero_remaining}")
print()
if zero_remaining == 0:
    print("All passages have at least 1 entity.")
else:
    print(f"WARNING: {zero_remaining} rows still have 0 entities!")